# Day 14 -- Fine-Tuning BERT
## Phase 3: NLP & Transformers

**Date:** April 24, 2026

### Learning Objectives

- Understand fine-tuning vs training from scratch
- Use HuggingFace Trainer API and TrainingArguments
- Load and prepare data with the `datasets` library
- Evaluate with `classification_report`

---
## Setup

In a real setup you would run:
```bash
pip install transformers datasets torch scikit-learn
```

In [ ]:
# Core imports
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
)
import warnings
warnings.filterwarnings("ignore")

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## Sample Data

We will create a small synthetic movie-review sentiment dataset inline. Label 0 = negative, label 1 = positive. This is tiny on purpose so it runs fast on CPU.

In [ ]:
# --- Synthetic movie review dataset ---
texts = [
    # Positive (label 1)
    "Absolutely loved this movie, a true masterpiece!",
    "The acting was superb and the plot kept me hooked.",
    "One of the best films I have seen this year.",
    "Brilliant cinematography and an unforgettable soundtrack.",
    "I laughed, I cried, I want to watch it again.",
    "A heartwarming story with fantastic performances.",
    "Every scene was beautifully crafted. Highly recommended.",
    "The director did an amazing job bringing the story to life.",
    "Such a fun and entertaining ride from start to finish.",
    "Great chemistry between the leads. Really enjoyable.",
    "This film deserves all the awards it can get.",
    "A feel-good movie that leaves you smiling.",
    "Incredible visuals and a touching story. Loved it.",
    "The script was clever and the pacing was perfect.",
    "I was on the edge of my seat the entire time.",
    "Wonderful performances from the entire cast.",
    "A delightful surprise. Much better than I expected.",
    "Funny, smart, and full of heart. Go see it.",
    "The best movie I have watched in a long time.",
    "An instant classic. Truly remarkable filmmaking.",
    "Beautiful storytelling and a powerful ending.",
    "So good I watched it twice in one weekend.",
    "A charming little film with a big message.",
    "Top-notch acting and a gripping narrative.",
    "I would give it six stars if I could.",
    # Negative (label 0)
    "What a waste of two hours. Terrible movie.",
    "The plot made no sense and the acting was wooden.",
    "I fell asleep halfway through. So boring.",
    "Possibly the worst film I have ever seen.",
    "The dialogue was cringeworthy and the effects were cheap.",
    "Predictable story with zero originality.",
    "I wanted to leave the theater after 20 minutes.",
    "Nothing about this movie worked. A total disaster.",
    "The pacing was painfully slow and the ending was weak.",
    "Awful acting and a script that felt like it was written in five minutes.",
    "A huge disappointment. Do not waste your money.",
    "The trailer was better than the entire movie.",
    "Overrated and overhyped. Skip this one.",
    "Confusing plot, flat characters, and bad music.",
    "I cannot believe this got a theatrical release.",
    "The worst sequel I have ever had to sit through.",
    "Dull, lifeless, and completely forgettable.",
    "Two thumbs way down. Save yourself the trouble.",
    "Uninspired directing and lazy storytelling.",
    "A mess from beginning to end. Just awful.",
    "The jokes did not land and the drama felt forced.",
    "Boring characters and a plot full of holes.",
    "I regret watching this. Total waste of time.",
    "Not even the popcorn could save this experience.",
    "An embarrassment to the franchise. Truly bad.",
]

labels = [1]*25 + [0]*25  # 25 positive, 25 negative

print(f"Total samples: {len(texts)}")
print(f"Positive: {sum(labels)}, Negative: {len(labels) - sum(labels)}")

In [ ]:
# Build a HuggingFace DatasetDict with train/test splits
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

raw_dataset = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts, "label": train_labels}),
    "test": Dataset.from_dict({"text": test_texts, "label": test_labels}),
})

print(raw_dataset)
print("\nSample row:", raw_dataset["train"][0])

---
## 1. What is Fine-Tuning?

Fine-tuning means taking a model that was **already trained** on a huge amount of text (like BERT, which was trained on Wikipedia and BookCorpus) and then training it a little bit more on **your specific task**. The model already understands language. You just teach it to apply that understanding to your problem.

This is very different from training from scratch. Training from scratch means starting with random weights and needing millions of examples. Fine-tuning needs far less data because the model already knows how language works.

Here is a simplified view of the process:

```
  PRETRAINED BERT (knows language)
         |
         v
  +--------------------+
  |  BERT Layers       |  <-- weights get slightly updated
  |  (12 transformer   |
  |   blocks)          |
  +--------------------+
         |
         v
  +--------------------+
  | Classification     |  <-- NEW layer, trained from scratch
  | Head (Linear)      |
  +--------------------+
         |
         v
     [positive / negative]
```

The key idea: BERT learns the language, the classification head learns the task.

---
## 2. Tokenization with AutoTokenizer

Before feeding text to BERT, we need to convert words into token IDs that the model understands. The `AutoTokenizer` handles this. It knows which vocabulary BERT uses and how to split words into subword tokens.

We use `padding=True` so all sequences in a batch have the same length, and `truncation=True` so nothing exceeds the model's max length (512 tokens for BERT).

In [ ]:
# Load tokenizer -- using distilbert for speed
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Let's see what tokenization looks like for a single sentence
sample = "I loved this movie!"
tokens = tokenizer(sample)

print("Input:", sample)
print("Token IDs:", tokens["input_ids"])
print("Attention mask:", tokens["attention_mask"])
print("Decoded back:", tokenizer.decode(tokens["input_ids"]))
print("Individual tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"]))

In [ ]:
# Tokenize the entire dataset using .map()
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64,  # short sentences, so 64 is plenty
    )

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)

# Remove the raw text column -- the model only needs input_ids, attention_mask, label
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Set format to PyTorch tensors
tokenized_dataset.set_format("torch")

print(tokenized_dataset)
print("\nColumns:", tokenized_dataset["train"].column_names)
print("Sample:", tokenized_dataset["train"][0])

---
## 3. AutoModelForSequenceClassification

This class loads a pretrained transformer and sticks a classification head on top. Under the hood it is just BERT (or DistilBERT) followed by a dropout layer and a linear layer that outputs `num_labels` logits.

You will see a warning that some weights are randomly initialized. That is expected. It is the classification head, which has never been trained before.

In [ ]:
# Load model with 2 labels (positive / negative)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Let's peek at the model architecture
print(model)

---
## 4. TrainingArguments

TrainingArguments controls everything about how training runs. Here are the key parameters:

| Parameter | What it does |
|---|---|
| `output_dir` | Where to save checkpoints |
| `num_train_epochs` | How many passes over the training data |
| `per_device_train_batch_size` | Batch size per device |
| `learning_rate` | Step size for the optimizer (use small values like 2e-5 for fine-tuning) |
| `evaluation_strategy` | When to evaluate ("epoch", "steps", or "no") |
| `save_strategy` | When to save checkpoints |
| `logging_steps` | How often to log training loss |
| `load_best_model_at_end` | Whether to reload the best checkpoint after training |
| `metric_for_best_model` | Which metric to use for "best" (e.g. "f1") |

In [ ]:
# Define training arguments
# NOTE: In real training you would use more epochs (3-5), larger batch sizes,
# and much more data. We keep it tiny here for fast CPU execution.

training_args = TrainingArguments(
    output_dir="./fine_tuned_model",
    num_train_epochs=1,                    # just 1 epoch for speed
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,                    # small LR for fine-tuning
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",                      # disable wandb/tensorboard logging
    no_cuda=True,                          # force CPU
)

print("TrainingArguments created.")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

---
## 5. compute_metrics Function

The Trainer can evaluate your model automatically, but you need to tell it *how* to compute metrics. You write a function that takes an `EvalPrediction` object (which has `.predictions` and `.label_ids`) and returns a dictionary of metric names and values.

In [ ]:
def compute_metrics(eval_pred: EvalPrediction) -> dict:
    """Compute accuracy, precision, recall, and F1 for binary classification."""
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

print("compute_metrics function defined.")

---
## 6. Trainer API

The Trainer class brings everything together. You give it the model, the training arguments, the datasets, the tokenizer, and the metrics function. Then you call `.train()` and it handles the training loop, logging, evaluation, and checkpointing for you.

No manual training loop needed. No `loss.backward()`. No `optimizer.step()`. The Trainer does it all.

In [ ]:
# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer created. Ready to train!")

In [ ]:
# Train the model
# This will take a minute or two on CPU with our tiny dataset
train_result = trainer.train()

print("\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")

In [ ]:
# Evaluate on the test set
eval_results = trainer.evaluate()

print("Evaluation results:")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

---
## 7. classification_report

After training, we can get predictions on the test set and print a detailed classification report using sklearn. This gives us per-class precision, recall, and F1.

In [ ]:
# Get predictions
predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Print classification report
label_names = ["negative", "positive"]
print(classification_report(true_labels, preds, target_names=label_names))

---
## 8. Saving and Loading

Once your model is trained, you want to save it so you can use it later without retraining. HuggingFace makes this simple with `save_pretrained()` and `from_pretrained()`.

In [ ]:
# Save the model and tokenizer
save_path = "./my_fine_tuned_sentiment_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved to {save_path}")

In [ ]:
# Load them back
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_path)
loaded_tokenizer = AutoTokenizer.from_pretrained(save_path)

# Quick test with the loaded model
test_text = "This movie was absolutely wonderful!"
inputs = loaded_tokenizer(test_text, return_tensors="pt", padding=True, truncation=True)

loaded_model.eval()
with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()

print(f"Text: {test_text}")
print(f"Prediction: {label_names[prediction]} (label {prediction})")

---
## Tricky Bits

Here are some common mistakes people make when fine-tuning.

### Mistake 1: Forgetting eval mode for inference

If you do not set the model to eval mode, dropout layers stay active and you get inconsistent predictions.

In [ ]:
# WRONG -- dropout is still active, results may vary each time
loaded_model.train()  # training mode!
inputs = loaded_tokenizer("Great movie!", return_tensors="pt")
with torch.no_grad():
    out1 = loaded_model(**inputs).logits
    out2 = loaded_model(**inputs).logits
print("Train mode - outputs may differ:")
print(f"  Run 1: {out1}")
print(f"  Run 2: {out2}")

# RIGHT -- set to eval mode
loaded_model.eval()
with torch.no_grad():
    out1 = loaded_model(**inputs).logits
    out2 = loaded_model(**inputs).logits
print("\nEval mode - outputs are consistent:")
print(f"  Run 1: {out1}")
print(f"  Run 2: {out2}")

### Mistake 2: Mismatched num_labels

If your data has 3 classes but you set `num_labels=2`, the model will crash or produce garbage.

In [ ]:
# Example: if you had 3 classes but loaded with num_labels=2
# The loss function would fail because label index 2 is out of range
try:
    bad_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    )
    fake_input = loaded_tokenizer("test", return_tensors="pt")
    fake_input["labels"] = torch.tensor([2])  # label 2 does not exist in a 2-class model!
    output = bad_model(**fake_input)
    print("This might error or give wrong loss")
except Exception as e:
    print(f"Error: {e}")

print("\nAlways make sure num_labels matches your actual number of classes!")

### Mistake 3: Not returning tensors from the tokenizer

When you manually run inference, you must pass `return_tensors="pt"` to the tokenizer. Without it, you get plain Python lists, and the model will not accept them.

In [ ]:
# WRONG -- no return_tensors
try:
    bad_inputs = loaded_tokenizer("test sentence")
    print(f"Type of input_ids: {type(bad_inputs['input_ids'])}")
    # This would fail: loaded_model(**bad_inputs)
    print("These are plain lists, not tensors. The model would reject them.")
except Exception as e:
    print(f"Error: {e}")

# RIGHT -- return_tensors="pt"
good_inputs = loaded_tokenizer("test sentence", return_tensors="pt")
print(f"\nType of input_ids: {type(good_inputs['input_ids'])}")
print("These are PyTorch tensors. The model will accept them.")

### Mistake 4: Forgetting to remove unused columns

If your dataset has columns like "text" that are not expected by the model, the Trainer might pass them to the model and cause errors. Always remove columns the model does not need.

In [ ]:
# The Trainer actually handles this automatically with remove_unused_columns=True (default)
# But if you set remove_unused_columns=False, you need to clean up manually

# Example of manual cleanup:
demo_ds = Dataset.from_dict({"text": ["hello"], "label": [1], "extra_col": ["junk"]})
print("Before:", demo_ds.column_names)

demo_ds = demo_ds.remove_columns(["extra_col"])
print("After:", demo_ds.column_names)

---
## Trick Questions

Test your understanding with these questions. Try to answer before revealing the solution.

---

**Q1: If you fine-tune BERT on sentiment analysis, can you then use it for NER (Named Entity Recognition) without retraining?**

<details>
<summary>Answer</summary>

No. Fine-tuning adds a task-specific head on top of BERT. A sentiment model has a classification head that outputs 2 classes (positive/negative). NER needs a completely different head that outputs a label for each token. You would need to start from the base BERT model again and fine-tune with a token classification head.
</details>

---

**Q2: Does fine-tuning update ALL of BERT's weights or just the classification head?**

<details>
<summary>Answer</summary>

By default, fine-tuning updates ALL weights, including BERT's internal transformer layers. However, the updates to BERT's layers are small because we use a very low learning rate. You can choose to freeze BERT's layers and only train the classification head, but this usually gives worse results because BERT cannot adapt its representations to your specific task.
</details>

---

**Q3: What happens if you set the learning rate too high when fine-tuning?**

<details>
<summary>Answer</summary>

The pretrained weights will get destroyed. BERT spent a long time learning useful language representations, and a high learning rate will overwrite them with noise. This is called "catastrophic forgetting." The model will perform poorly because it lost the language knowledge it started with. That is why fine-tuning uses learning rates like 2e-5 or 5e-5, much smaller than the 1e-3 commonly used for training from scratch.
</details>

---

**Q4: Why do we use a smaller learning rate for fine-tuning than for training from scratch?**

<details>
<summary>Answer</summary>

Because the pretrained weights are already in a good region of the loss landscape. We want to nudge them slightly toward our task, not jump to a completely different region. A large learning rate would push the weights too far and destroy the useful features BERT already learned. Think of it like adjusting a well-tuned radio dial: you only need tiny movements.
</details>

---
## Exercises

Fill in the blanks (`___`) and make sure the asserts pass.

In [ ]:
# Exercise 1: Load a tokenizer for "distilbert-base-uncased"

ex1_tokenizer = ___.from_pretrained(___)

assert ex1_tokenizer.vocab_size == 30522, "Should be distilbert-base-uncased vocab size"
print("Exercise 1 passed!")

In [ ]:
# Exercise 2: Tokenize a list of sentences with padding and truncation

sentences = ["I love NLP", "Transformers are cool"]
ex2_output = ex1_tokenizer(
    sentences,
    padding=___,
    truncation=___,
    return_tensors=___,
)

assert ex2_output["input_ids"].shape[0] == 2, "Should have 2 sequences"
assert isinstance(ex2_output["input_ids"], torch.Tensor), "Should be PyTorch tensors"
print("Exercise 2 passed!")

In [ ]:
# Exercise 3: Create TrainingArguments with 2 epochs and batch size 16

ex3_args = TrainingArguments(
    output_dir="./ex3_output",
    num_train_epochs=___,
    per_device_train_batch_size=___,
    report_to="none",
)

assert ex3_args.num_train_epochs == 2, "Should be 2 epochs"
assert ex3_args.per_device_train_batch_size == 16, "Should be batch size 16"
print("Exercise 3 passed!")

In [ ]:
# Exercise 4: Write a compute_metrics function that returns accuracy

def ex4_compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=___)
    acc = ___(labels, preds)
    return {"accuracy": acc}

# Test it with fake data
fake_logits = np.array([[0.1, 0.9], [0.8, 0.2], [0.3, 0.7]])
fake_labels = np.array([1, 0, 1])
fake_eval = EvalPrediction(predictions=fake_logits, label_ids=fake_labels)
result = ex4_compute_metrics(fake_eval)

assert result["accuracy"] == 1.0, "All predictions should be correct"
print("Exercise 4 passed!")

In [ ]:
# Exercise 5: Load a model with the right number of labels for 3-class classification

ex5_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=___,
)

assert ex5_model.config.num_labels == 3, "Should have 3 labels"
print("Exercise 5 passed!")

In [ ]:
# Exercise 6: Given predictions array, compute accuracy manually

ex6_logits = np.array([[2.0, -1.0], [-1.0, 2.0], [0.5, -0.5], [-0.5, 0.5]])
ex6_labels = np.array([0, 1, 0, 1])

ex6_preds = np.argmax(ex6_logits, axis=___)
ex6_accuracy = ___(ex6_labels, ex6_preds)

assert ex6_accuracy == 1.0, "All predictions match labels"
print("Exercise 6 passed!")

---
## Solutions

<details>
<summary>Click to reveal all solutions</summary>

**Exercise 1:**
```python
ex1_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
```

**Exercise 2:**
```python
ex2_output = ex1_tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
```

**Exercise 3:**
```python
ex3_args = TrainingArguments(
    output_dir="./ex3_output",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    report_to="none",
)
```

**Exercise 4:**
```python
preds = np.argmax(logits, axis=1)
acc = accuracy_score(labels, preds)
```

**Exercise 5:**
```python
ex5_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
)
```

**Exercise 6:**
```python
ex6_preds = np.argmax(ex6_logits, axis=1)
ex6_accuracy = accuracy_score(ex6_labels, ex6_preds)
```
</details>

---
## Cumulative Review Exercises (Days 4-13)

Mixed exercises covering topics from the last 10 days. Fill in the blanks and make sure asserts pass.

In [ ]:
# Review 1 (Day 4 - Faker, list comprehensions)
# Create a list of 5 squared numbers using a list comprehension

squares = [x ___ 2 for x in range(1, 6)]

assert squares == [1, 4, 9, 16, 25], "Should be squares of 1 through 5"
print("Review 1 passed!")

In [ ]:
# Review 2 (Day 5 - PyTorch tensors)
# Create a 3x3 tensor of ones and compute its sum

ones = torch.___(3, 3)
total = ones.___().item()

assert total == 9.0, "Sum of 3x3 ones tensor should be 9"
print("Review 2 passed!")

In [ ]:
# Review 3 (Day 6 - train_test_split)
from sklearn.model_selection import train_test_split

data = list(range(100))
labels_r3 = [0]*50 + [1]*50

X_train, X_test, y_train, y_test = train_test_split(
    data, labels_r3, test_size=___, random_state=42
)

assert len(X_test) == 20, "20% of 100 should be 20"
print("Review 3 passed!")

In [ ]:
# Review 4 (Day 7 - LogisticRegression)
from sklearn.linear_model import LogisticRegression

lr_model = ___(max_iter=200)

assert hasattr(lr_model, "fit"), "Should be a sklearn estimator with a fit method"
assert hasattr(lr_model, "predict"), "Should have a predict method"
print("Review 4 passed!")

In [ ]:
# Review 5 (Day 8 - RandomForest)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=___, random_state=42)

assert rf.n_estimators == 50, "Should have 50 trees"
print("Review 5 passed!")

In [ ]:
# Review 6 (Day 9 - confusion_matrix)
from sklearn.metrics import confusion_matrix

y_true_r6 = [1, 0, 1, 1, 0, 0, 1, 0]
y_pred_r6 = [1, 0, 1, 0, 0, 1, 1, 0]

cm = ___(y_true_r6, y_pred_r6)

assert cm.shape == (2, 2), "Should be a 2x2 matrix"
assert cm[0][0] == 3, "True negatives should be 3"
print("Review 6 passed!")

In [ ]:
# Review 7 (Day 10 - SHAP concepts)
# SHAP values explain feature importance. Fill in the concept.

shap_concept = "___"  # What does SHAP stand for? Fill in: "SHapley Additive exPlanations"

assert "shapley" in shap_concept.lower(), "Should contain 'Shapley'"
assert "explanations" in shap_concept.lower(), "Should contain 'Explanations'"
print("Review 7 passed!")

In [ ]:
# Review 8 (Day 11 - TfidfVectorizer)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = ___(max_features=100)
sample_corpus = ["I love NLP", "NLP is great", "I love transformers"]
tfidf_matrix = vectorizer.fit_transform(sample_corpus)

assert tfidf_matrix.shape[0] == 3, "Should have 3 documents"
assert tfidf_matrix.shape[1] <= 100, "Should have at most 100 features"
print("Review 8 passed!")

In [ ]:
# Review 9 (Day 12 - Embeddings concepts)
# Word embeddings map words to dense vectors. Fill in the dimension.

# A typical Word2Vec embedding dimension
embedding_dim = ___  # Common values: 50, 100, 200, 300

assert embedding_dim in [50, 100, 200, 300], "Should be a common embedding dimension"
print("Review 9 passed!")

In [ ]:
# Review 10 (Day 13 - HuggingFace pipeline and AutoTokenizer)
from transformers import AutoTokenizer

rev10_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
rev10_tokens = rev10_tokenizer("Hello world", return_tensors=___)

assert isinstance(rev10_tokens["input_ids"], torch.Tensor), "Should return PyTorch tensors"
print("Review 10 passed!")

---
### Cumulative Review Solutions

<details>
<summary>Click to reveal all solutions</summary>

**Review 1:** `x ** 2`

**Review 2:** `torch.ones(3, 3)` and `ones.sum().item()`

**Review 3:** `test_size=0.2`

**Review 4:** `LogisticRegression(max_iter=200)`

**Review 5:** `n_estimators=50`

**Review 6:** `confusion_matrix(y_true_r6, y_pred_r6)`

**Review 7:** `"SHapley Additive exPlanations"`

**Review 8:** `TfidfVectorizer(max_features=100)`

**Review 9:** Any of `50`, `100`, `200`, or `300`

**Review 10:** `return_tensors="pt"`
</details>

---
## Cheat Sheet

In [ ]:
cheat_sheet = """
============================================================
   FINE-TUNING BERT -- QUICK REFERENCE CARD
============================================================

LOADING
-------
  tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
  model     = AutoModelForSequenceClassification.from_pretrained(
                  "distilbert-base-uncased", num_labels=2)

TOKENIZATION
------------
  tokens = tokenizer(text, padding=True, truncation=True,
                     return_tensors="pt", max_length=512)
  # For datasets: dataset.map(tokenize_fn, batched=True)

TRAINING ARGUMENTS (typical fine-tuning values)
------------------------------------------------
  TrainingArguments(
      output_dir="./results",
      num_train_epochs=3,          # 2-5 epochs
      per_device_train_batch_size=16,
      learning_rate=2e-5,          # 1e-5 to 5e-5
      weight_decay=0.01,
      evaluation_strategy="epoch",
      save_strategy="epoch",
      load_best_model_at_end=True,
      metric_for_best_model="f1",
  )

TRAINER
-------
  trainer = Trainer(
      model=model, args=args,
      train_dataset=train_ds, eval_dataset=eval_ds,
      tokenizer=tokenizer,
      compute_metrics=compute_metrics,
  )
  trainer.train()
  trainer.evaluate()
  preds = trainer.predict(test_ds)

COMPUTE METRICS
---------------
  def compute_metrics(eval_pred):
      logits, labels = eval_pred.predictions, eval_pred.label_ids
      preds = np.argmax(logits, axis=1)
      p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
      return {"accuracy": accuracy_score(labels, preds),
              "precision": p, "recall": r, "f1": f1}

SAVING / LOADING
----------------
  model.save_pretrained("./my_model")
  tokenizer.save_pretrained("./my_model")
  loaded = AutoModelForSequenceClassification.from_pretrained("./my_model")

INFERENCE
---------
  model.eval()  # IMPORTANT!
  with torch.no_grad():
      inputs = tokenizer(text, return_tensors="pt")
      outputs = model(**inputs)
      pred = torch.argmax(outputs.logits, dim=1)

CLASSIFICATION REPORT
---------------------
  from sklearn.metrics import classification_report
  print(classification_report(y_true, y_pred, target_names=["neg", "pos"]))

============================================================
"""
print(cheat_sheet)

---

**Next up: Day 15 -- Complaint Classification Project (Turkish complaint classification mini project)**